In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

HERE = Path('.')

# ── Retenções FB-14 ───────────────────────────────────────────────────────────
df_raw = pd.read_csv(HERE / 'retidos_sql.csv', header=None)
df_raw.columns = ['data','linha','maquina','cod_produto','descricao','quantidade','motivo','observacao']
df_raw['data'] = pd.to_datetime(df_raw['data'], errors='coerce', utc=True)

fb14 = df_raw[df_raw['maquina'].astype(str).str.contains(r'FB-?14', case=False, na=False)].copy()
fb14_2024 = fb14[fb14['data'].dt.year == 2024].copy()

# ── Pipeline (score + lifecycle) ──────────────────────────────────────────────
df7 = pd.read_csv(HERE / '07_pipeline_v2.csv')
df7['ts'] = pd.to_datetime(df7['Timestamp'], utc=True)

# ── Trocas de maintacker ──────────────────────────────────────────────────────
trocas_raw = pd.read_csv(HERE / 'troca_modulo.csv', encoding='utf-8-sig')
trocas = pd.to_datetime(trocas_raw.iloc[:, 0], utc=True).sort_values()
trocas_2024 = trocas[trocas.dt.year == 2024]

print(f"Retenções FB-14 em 2024 : {len(fb14_2024)} eventos")
print(f"Trocas de maintacker 2024: {len(trocas_2024)} eventos ({', '.join(trocas_2024.dt.strftime('%d/%m').tolist())})")
print(f"Pipeline 2024           : {(df7['ts'].dt.year == 2024).sum()} pontos horários")
print()
fb14_2024[['data','quantidade','motivo','observacao']]

In [ ]:
# ── Contexto de score e lifecycle para cada evento de retenção ───────────────
JANELA_PRE = 14   # dias antes do evento para calcular score médio

registros = []
for _, ev in fb14_2024.iterrows():
    t = ev['data']
    pre = df7[(df7['ts'] >= t - pd.Timedelta(days=JANELA_PRE)) & (df7['ts'] < t)]
    no_dia = df7[df7['ts'].dt.date == t.date()]

    score_pre  = pre['score_degradacao_v2'].mean() if len(pre) else np.nan
    score_dia  = no_dia['score_degradacao_v2'].mean() if len(no_dia) else np.nan
    horas_ciclo = pre['horas_desde_troca'].mean() if len(pre) else np.nan
    pct_critico = (pre['score_degradacao_v2'] >= 0.60).mean() if len(pre) else np.nan

    # Troca mais próxima após o evento
    proximas = trocas[trocas > t]
    dias_ate_troca = (proximas.iloc[0] - t).days if len(proximas) else None

    registros.append({
        'data'           : t.strftime('%d/%m/%Y'),
        'tipo'           : 'Vazamento barreira' if 'azamento' in str(ev['observacao']) else 'Fardo aberto',
        'qtd_retida'     : int(ev['quantidade']) if pd.notna(ev['quantidade']) else 0,
        'score_14d_antes': round(score_pre, 3) if pd.notna(score_pre) else None,
        'score_no_dia'   : round(score_dia, 3) if pd.notna(score_dia) else None,
        'pct_dias_critico': f"{pct_critico*100:.0f}%" if pd.notna(pct_critico) else None,
        'horas_ciclo'    : round(horas_ciclo) if pd.notna(horas_ciclo) else None,
        'dias_ate_proxima_troca': dias_ate_troca,
    })

resumo = pd.DataFrame(registros).drop_duplicates(subset=['data','tipo'])
print("=== Eventos de retenção FB-14 2024 × Score × Lifecycle ===\n")
print(resumo.to_string(index=False))

In [ ]:
# ── Gráfico: score + lifecycle + eventos de retenção + trocas (2024) ─────────
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
fig.patch.set_facecolor('#0f1923')
for ax in axes:
    ax.set_facecolor('#162030')
    ax.tick_params(colors='#5a7080')
    ax.spines[:].set_color('#243347')

df24 = df7[df7['ts'].dt.year == 2024].copy()
ts   = df24['ts']

# Painel 1 — Score de degradação
score = df24['score_degradacao_v2'].fillna(0)
colors = ['#e74c3c' if s >= 0.60 else '#e67e22' if s >= 0.30 else '#27ae60' for s in score]
axes[0].bar(ts, score, color=colors, alpha=0.8, width=0.04)
axes[0].axhline(0.60, color='#e74c3c', ls='--', lw=1.2, alpha=0.7, label='Limiar crítico (0,60)')
axes[0].axhline(0.30, color='#e67e22', ls=':', lw=1.0, alpha=0.5)
axes[0].set_ylabel('Score Degradação', color='#5a7080', fontsize=9)
axes[0].set_ylim(0, 1.05)
axes[0].legend(fontsize=8, facecolor='#162030', labelcolor='#e8eef5')

# Painel 2 — Horas no ciclo
axes[1].plot(ts, df24['horas_desde_troca'], color='#3498db', lw=1.2)
axes[1].fill_between(ts, df24['horas_desde_troca'], alpha=0.15, color='#3498db')
axes[1].set_ylabel('Horas desde troca', color='#5a7080', fontsize=9)

# Linha da vida mediana (carregada do weibull params se disponível)
try:
    import json
    with open(HERE / '06_weibull_params.json') as f:
        wp = json.load(f)
    vida_med = wp['vida_mediana_h']
    axes[1].axhline(vida_med, color='#e67e22', ls='--', lw=1.0, alpha=0.6,
                    label=f'Vida mediana ({vida_med:.0f}h)')
    axes[1].legend(fontsize=8, facecolor='#162030', labelcolor='#e8eef5')
except Exception:
    pass

# Painel 3 — Força de selagem média
axes[2].plot(ts, df24['Media'], color='#3498db', lw=0.8, alpha=0.7)
axes[2].axhline(800, color='#e74c3c', ls=':', lw=1.0, alpha=0.5, label='Limiar 800N')
axes[2].set_ylabel('Força Média (N)', color='#5a7080', fontsize=9)
axes[2].legend(fontsize=8, facecolor='#162030', labelcolor='#e8eef5')

# Marcadores: trocas de maintacker
for t_troca in trocas_2024:
    for ax in axes:
        ax.axvline(t_troca, color='#FFB347', lw=1.2, alpha=0.7)

# Marcadores: eventos de retenção
CORES_RET = {'Vazamento barreira': '#e74c3c', 'Fardo aberto': '#9b59b6'}
datas_ret_plotadas = set()
for _, ev in fb14_2024.iterrows():
    tipo = 'Vazamento barreira' if 'azamento' in str(ev['observacao']) else 'Fardo aberto'
    key  = (ev['data'].date(), tipo)
    if key in datas_ret_plotadas:
        continue
    datas_ret_plotadas.add(key)
    for ax in axes:
        ax.axvline(ev['data'], color=CORES_RET[tipo], lw=2.0, ls='--', alpha=0.9)

# Legenda manual
patches = [
    mpatches.Patch(color='#FFB347',  label='Troca maintacker'),
    mpatches.Patch(color='#e74c3c',  label='Retenção: Vazamento barreira'),
    mpatches.Patch(color='#9b59b6',  label='Retenção: Fardo aberto'),
]
axes[0].legend(handles=patches + [mpatches.Patch(color='#e74c3c', alpha=0, label='Limiar crítico (0,60) ──')],
               fontsize=8, facecolor='#162030', labelcolor='#e8eef5')

fig.suptitle('FB-14 2024 — Score de Degradação × Retenções × Trocas de Maintacker',
             color='#e8eef5', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('retidos_cruzamento_2024.png', dpi=130, bbox_inches='tight', facecolor='#0f1923')
plt.show()

In [ ]:
# ── Narrativa gerada automaticamente dos dados ────────────────────────────────
import json

with open(HERE / '06_weibull_params.json') as f:
    wp = json.load(f)
vida_med_h = wp['vida_mediana_h']

linhas = ["=" * 68]
linhas.append("  EVIDÊNCIA HISTÓRICA — Retenções × Score × Lifecycle (FB-14 2024)")
linhas.append("=" * 68)

for _, row in resumo.iterrows():
    score_pre = row['score_14d_antes']
    horas     = row['horas_ciclo']
    dias_troca = row['dias_ate_proxima_troca']
    pct_vida  = (horas / vida_med_h * 100) if horas else 0
    status_score = "CRÍTICO" if score_pre and score_pre >= 0.60 else "ATENÇÃO" if score_pre and score_pre >= 0.30 else "NORMAL"
    status_vida  = "fim de ciclo" if pct_vida >= 70 else "meio de ciclo" if pct_vida >= 40 else "início de ciclo"

    linhas.append(f"\n  Evento: {row['tipo']} — {row['data']}")
    linhas.append(f"  ├─ Quantidade retida   : {row['qtd_retida']:,} unidades")
    linhas.append(f"  ├─ Score (14d antes)   : {score_pre} → {status_score}")
    linhas.append(f"  ├─ Dias em CRÍTICO (14d): {row['pct_dias_critico']}")
    linhas.append(f"  ├─ Horas no ciclo      : {horas:.0f}h ({pct_vida:.0f}% da vida mediana) → {status_vida}")
    if dias_troca is not None:
        linhas.append(f"  └─ Troca seguinte      : {dias_troca} dias depois")
    else:
        linhas.append(f"  └─ Troca seguinte      : não identificada")

linhas.append("\n" + "=" * 68)
linhas.append("  INTERPRETAÇÃO")
linhas.append("=" * 68)

# Evento Junho - vazamento barreira (o mais crítico)
ev_jun = resumo[resumo['tipo'] == 'Vazamento barreira']
if len(ev_jun):
    r = ev_jun.iloc[0]
    pct = (r['horas_ciclo'] / vida_med_h * 100)
    linhas.append(f"""
  → {r['tipo']} ({r['data']}):
    Score médio nos 14 dias anteriores: {r['score_14d_antes']} (acima do limiar crítico de 0,60).
    O modelo estava em CRÍTICO por {r['pct_dias_critico']} dos dias antes do evento.
    O maintacker operava com {r['horas_ciclo']:.0f}h ({pct:.0f}% da vida mediana).
    DIAGNÓSTICO: desgaste mecânico. O sinal de degradação antecedeu o evento de retenção.
    Troca realizada {r['dias_ate_proxima_troca']} dias após a retenção.""")

# Evento Janeiro - fardo aberto
ev_jan = resumo[resumo['tipo'] == 'Fardo aberto']
if len(ev_jan):
    r = ev_jan.iloc[0]
    pct = (r['horas_ciclo'] / vida_med_h * 100)
    linhas.append(f"""
  → {r['tipo']} ({r['data']}):
    Score médio nos 14 dias anteriores: {r['score_14d_antes']} (abaixo do limiar crítico).
    O maintacker operava com {r['horas_ciclo']:.0f}h ({pct:.0f}% da vida mediana — início de ciclo).
    DIAGNÓSTICO: provável problema operacional (adesivo/ajuste). Peça ainda nova.
    Troca realizada {r['dias_ate_proxima_troca']} dias após → pode ter sido desnecessária.""")

linhas.append("\n" + "=" * 68)
print("\n".join(linhas))